In [ ]:
print("hello")

hello


In [ ]:
from sklearn.model_selection import train_test_split

# Xác định target column dựa trên schema hoặc phân tích dữ liệu
# Giả định cột cuối cùng hoặc cột target cụ thể từ schema là nhãn cần dự đoán.
# Ở bước này, chúng ta sẽ tự động xác định target column phù hợp.
target_col = None
if 'target' in df.columns:
    target_col = 'target'
else:
    # Nếu không có cột tên 'target', ta lấy cột cuối cùng làm nhãn dự đoán mặc định
    target_col = df.columns[-1]

print(f"Cột mục tiêu được xác định là: {target_col}")

X = df.drop(columns=[target_col])
y = df[target_col]

# Chia tập Train và Test (Giữ nguyên tập test để đánh giá cuối cùng)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y if y.dtype == 'object' or y.nunique() < 10 else None)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Cột mục tiêu được xác định là: loan_status
Kích thước tập Train: (491, 12)
Kích thước tập Test: (123, 12)


In [ ]:
# Tải preprocessor pipeline đã được lưu sẵn
try:
    preprocessor = joblib.load('/content/preprocessor_pipeline.joblib')
    print("Đã tải thành công preprocessor_pipeline.joblib")

    # Áp dụng tiền xử lý lên tập train và test
    X_train_processed = preprocessor.fit_transform(X_train, y_train) if hasattr(preprocessor, 'fit_transform') else preprocessor.transform(X_train)
    X_test_processed = preprocessor.transform(X_test)
except Exception as e:
    print(f"Không thể áp dụng preprocessor gốc ({e}). Tiến hành xử lý tự động cơ bản:")
    # Hướng xử lý dự phòng nếu pipeline không tương thích trực tiếp
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline

    num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X_train.select_dtypes(exclude=['int64', 'float64']).columns

    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)

print("Xử lý dữ liệu hoàn tất.")

Đã tải thành công preprocessor_pipeline.joblib
Xử lý dữ liệu hoàn tất.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import classification_report, accuracy_score, f1_score, mean_squared_error, r2_score
import numpy as np

# Xác định bài toán là Phân loại (Classification) hay Hồi quy (Regression)
is_classification = y.dtype == 'object' or y.nunique() < 10

# Định nghĩa các mô hình và không gian siêu tham số tương ứng
if is_classification:
    print("--- Đang thực hiện bài toán PHÂN LOẠI (Classification) ---")

    # 1. Baseline Model
    baseline = DummyClassifier(strategy='most_frequent')
    baseline.fit(X_train_processed, y_train)
    baseline_pred = baseline.predict(X_test_processed)
    print(f"Baseline Accuracy: {accuracy_score(y_test, baseline_pred):.4f}")

    # Định nghĩa cấu hình các mô hình còn lại
    models_config = {
        'Logistic Regression': {
            'model': LogisticRegression(max_iter=1000, random_state=42),
            'params': {'C': [0.1, 1, 10]}
        },
        'Decision Tree': {
            'model': DecisionTreeClassifier(random_state=42),
            'params': {'max_depth': [3, 5, 10], 'min_samples_split': [2, 5]}
        },
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {'n_estimators': [50, 100], 'max_depth': [5, 10]}
        },
        'SVM': {
            'model': SVC(random_state=42),
            'params': {'C': [1, 10], 'kernel': ['linear', 'rbf']}
        }
    }

    # Train & Tinh chỉnh siêu tham số với GridSearchCV
    best_models = {}
    for name, config in models_config.items():
        print(f"\nĐang tinh chỉnh {name}...")
        grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='accuracy', n_jobs=-1)
        grid.fit(X_train_processed, y_train)
        best_models[name] = grid.best_estimator_
        print(f"Tham số tốt nhất cho {name}: {grid.best_params_}")
        print(f"Accuracy cao nhất trên tập Train (CV): {grid.best_score_:.4f}")

else:
    print("--- Đang thực hiện bài toán HỒI QUY (Regression) ---")

    # 1. Baseline Model
    baseline = DummyRegressor(strategy='mean')
    baseline.fit(X_train_processed, y_train)
    baseline_pred = baseline.predict(X_test_processed)
    print(f"Baseline MSE: {mean_squared_error(y_test, baseline_pred):.4f}")

    # Định nghĩa cấu hình các mô hình còn lại
    models_config = {
        'Ridge Regression': {
            'model': Ridge(),
            'params': {'alpha': [0.1, 1.0, 10.0]}
        },
        'Decision Tree Regressor': {
            'model': DecisionTreeRegressor(random_state=42),
            'params': {'max_depth': [3, 5, 10]}
        },
        'Random Forest Regressor': {
            'model': RandomForestClassifier(random_state=42), # Chuyển đổi linh hoạt nếu cần
            'params': {'n_estimators': [50, 100], 'max_depth': [5, 10]}
        },
        'SVR': {
            'model': SVR(),
            'params': {'C': [1, 10], 'kernel': ['linear', 'rbf']}
        }
    }

    # Train & Tinh chỉnh siêu tham số với GridSearchCV
    best_models = {}
    for name, config in models_config.items():
        print(f"\nĐang tinh chỉnh {name}...")
        grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        grid.fit(X_train_processed, y_train)
        best_models[name] = grid.best_estimator_
        print(f"Tham số tốt nhất cho {name}: {grid.best_params_}")
        print(f"MSE tốt nhất trên tập Train (CV): {-grid.best_score_:.4f}")

--- Đang thực hiện bài toán PHÂN LOẠI (Classification) ---
Baseline Accuracy: 0.6911

Đang tinh chỉnh Logistic Regression...
Tham số tốt nhất cho Logistic Regression: {'C': 0.1}
Accuracy cao nhất trên tập Train (CV): 0.7984

Đang tinh chỉnh Decision Tree...
Tham số tốt nhất cho Decision Tree: {'max_depth': 3, 'min_samples_split': 2}
Accuracy cao nhất trên tập Train (CV): 0.7861

Đang tinh chỉnh Random Forest...
Tham số tốt nhất cho Random Forest: {'max_depth': 5, 'n_estimators': 50}
Accuracy cao nhất trên tập Train (CV): 0.8004

Đang tinh chỉnh SVM...
Tham số tốt nhất cho SVM: {'C': 1, 'kernel': 'linear'}
Accuracy cao nhất trên tập Train (CV): 0.7984


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# 1. Định nghĩa Baseline và 4 mô hình theo yêu cầu
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train_processed, y_train)
baseline_pred = baseline.predict(X_test_processed)

models_config = {
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 9],
            'weights': ['uniform', 'distance']
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.1, 1, 10]
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [3, 5, 10, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'Naive Bayes': {
        'model': GaussianNB(),
        'params': {} # Naive Bayes thường không cần tinh chỉnh tham số phức tạp
    }
}

# 2. Huấn luyện và tìm kiếm siêu tham số tối ưu
best_models = {}
for name, config in models_config.items():
    print(f"Đang tối ưu hóa {name}...")
    grid = GridSearchCV(config['model'], config['params'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train_processed, y_train)
    best_models[name] = grid.best_estimator_
    print(f"-> Tham số tốt nhất: {grid.best_params_}")
    print(f"-> Accuracy (CV): {grid.best_score_:.4f}\n")

Đang tối ưu hóa KNN...
-> Tham số tốt nhất: {'n_neighbors': 9, 'weights': 'uniform'}
-> Accuracy (CV): 0.7902

Đang tối ưu hóa Logistic Regression...
-> Tham số tốt nhất: {'C': 0.1}
-> Accuracy (CV): 0.7984

Đang tối ưu hóa Decision Tree...
-> Tham số tốt nhất: {'max_depth': 3, 'min_samples_split': 2}
-> Accuracy (CV): 0.7861

Đang tối ưu hóa Naive Bayes...
-> Tham số tốt nhất: {}
-> Accuracy (CV): 0.7596



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Khởi tạo từ điển lưu kết quả đánh giá mới
evaluation_results = {}

# Định nghĩa danh sách các mô hình cần đánh giá
models_to_evaluate = {
    'Baseline (Most Frequent)': (None, baseline_pred),
    'KNN': (best_models['KNN'], None),
    'Logistic Regression': (best_models['Logistic Regression'], None),
    'Decision Tree': (best_models['Decision Tree'], None),
    'Naive Bayes': (best_models['Naive Bayes'], None)
}

for name, (model, preds) in models_to_evaluate.items():
    if preds is None:
        preds = model.predict(X_test_processed)

    # Tính toán các chỉ số (Sử dụng average='macro' cho trường hợp phân loại nhiều lớp hoặc không cân bằng)
    evaluation_results[name] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds, average='macro', zero_division=0),
        'Recall': recall_score(y_test, preds, average='macro', zero_division=0),
        'F1-score': f1_score(y_test, preds, average='macro', zero_division=0)
    }

# Chuyển đổi sang DataFrame để hiển thị đẹp mắt
full_report_df = pd.DataFrame(evaluation_results).T
print("--- BẢNG ĐÁNH GIÁ CHI TIẾT CÁC METRIC TRÊN TẬP TEST ---")
display(full_report_df)

--- BẢNG ĐÁNH GIÁ CHI TIẾT CÁC METRIC TRÊN TẬP TEST ---


,Accuracy,Precision,Recall,F1-score
Baseline (Most Frequent),0.691057,0.345528,0.500000,0.408654
KNN,0.837398,0.840603,0.765944,0.788805
Logistic Regression,0.861789,0.898261,0.783591,0.814710
Decision Tree,0.845528,0.887955,0.757276,0.788181
Naive Bayes,0.845528,0.858367,0.771827,0.797294
